In [1]:
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import r2_score, mean_squared_error

# 1. Load the cleaned dataset from Phase 1
df = pd.read_csv("cleaned_employee_dataset.csv")

# 2. Separate the clues (X) from the target answer (Salary)
X = df.drop("Salary_INR", axis=1) # Make sure "Salary_INR" matches your exact column name
y = df["Salary_INR"]

# 3. Split into 80% training data and 20% testing data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Selecting the top 8 most useful features...")
# 4. Use Forward Feature Selection to find the best 8 clues
base_model = LinearRegression()
selector = SequentialFeatureSelector(base_model, n_features_to_select=8, direction="forward")
X_train_selected = selector.fit_transform(X_train, y_train)
X_test_selected = selector.transform(X_test)

# Save the names of the 8 features so we know what to type in the input file later
selected_features = list(X.columns[selector.get_support()])
print(f"Features selected: {selected_features}\n")

print("Training the KNN Regression Model...")
# 5. Create and train the KNN brain (K=5)
knn_model = KNeighborsRegressor(n_neighbors=5)
knn_model.fit(X_train_selected, y_train)

# 6. Make guesses on the test data and grade the model
predictions = knn_model.predict(X_test_selected)
r2 = r2_score(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("--- KNN Model Results ---")
print(f"R2 Score: {r2:.4f} (Closer to 1.0 is better)")
print(f"RMSE    : ₹ {rmse:,.2f} (Average prediction mistake)")

# 7. Save the trained model and feature list for our Interactive Input file
with open('knn_salary_model.pkl', 'wb') as f:
    pickle.dump({
        'model': knn_model, 
        'selector': selector, 
        'feature_names': selected_features
    }, f)
print("\nSuccess! KNN Model saved as 'knn_salary_model.pkl'.")

Selecting the top 8 most useful features...
Features selected: ['Age', 'Experience', 'JobRole', 'PerformanceRating', 'ProjectsCompleted', 'LeadershipScore', 'Promotions', 'LocationType']

Training the KNN Regression Model...
--- KNN Model Results ---
R2 Score: 0.9087 (Closer to 1.0 is better)
RMSE    : ₹ 151,151.41 (Average prediction mistake)

Success! KNN Model saved as 'knn_salary_model.pkl'.
